In [1]:
from geopandas import GeoDataFrame, points_from_xy
import json
import os
import pandas as pd
import sqlite3

In [5]:
x = "a,b,c,d,e,f,g,h;a1,b1,c1,d1,e1,f1,g1,h1,i1,j1;a2,b2,c2,d2,e2,f2"
x = "a,b,c,d,e,f,g,h"
x1 = x.split(';')
x2 = []
for i in x1:
    x2.append(i.split(','))
#x2 = x1.split(',')
print(x2)

[['a', 'b', 'c', 'd', 'e', 'f', 'g', 'h']]


In [2]:
from datetime import datetime
now = datetime.now()
dt_string = now.strftime("%Y%m%d_%H%M%S")

print(dt_string)

20241028_115638


In [3]:
targetdir = "C:/IMAGE/Foliage/image_api/subgrids/"
filelist = os.listdir(targetdir)

In [4]:
def remove_substrings_from_end(string, substring_list):
    for substring in substring_list:
        if string.endswith(substring):
            return string[:-len(substring)]
        return string

In [5]:
df_list = [pd.read_table(targetdir+file) for file in filelist]
gdf_list = [GeoDataFrame(df, geometry=points_from_xy(df.PT_LON, df.PT_LAT), crs="EPSG:4326") for df in df_list]

In [6]:
dict_centre = {
                "area": ["bangalore", "delhi", "mumbai"],
                "centre_lat": [],
                "centre_lon": []
               }

for gdf, file in zip(gdf_list, filelist):
    file_name = remove_substrings_from_end(file, [".txt",".csv",".xlsx", ".xlsm"])
    dict_centre["centre_lat"].append(gdf["PT_LAT"].mean())
    dict_centre["centre_lon"].append(gdf["PT_LON"].mean())
    #dict_centre["centre"].append((gdf["PT_LON"].mean(), gdf["PT_LAT"].mean()))

print(dict_centre)

{'area': ['bangalore', 'delhi', 'mumbai'], 'centre_lat': [12.97110872536232, 28.641012270255928, 19.10130078932353], 'centre_lon': [77.59927757430772, 77.09628905684609, 72.95113610941772]}


In [7]:
df = pd.DataFrame.from_dict(dict_centre, orient="columns")
df

,area,centre_lat,centre_lon
0,bangalore,12.971109,77.599278
1,delhi,28.641012,77.096289
2,mumbai,19.101301,72.951136


In [8]:
conndir = 'C:/IMAGE/Foliage/image_api/data/subgrids.db'

conn = sqlite3.connect(conndir)
print(conn)
print(type(conn))
name = 'cluster_centres'

df.to_sql(
            name=name,
            con=conn,
            index=False,
            if_exists='append',
         )
conn.commit()
conn.close()

<class 'sqlite3.Connection'>


In [9]:
#with open('dataset_centroids.json', 'w') as json_file:
#    json.dump(dict_centre, json_file, indent=4)